Fig4C, cache builder

In [ ]:
# Notebook path bootstrap for moved files
import os, sys

def find_project_root(start_dir):
    cur = os.path.abspath(start_dir)
    while True:
        if os.path.isdir(os.path.join(cur, 'figs-submit')):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            raise FileNotFoundError('Could not find project root containing figs-submit.')
        cur = parent

PROJECT_ROOT = find_project_root(os.getcwd())
os.chdir(PROJECT_ROOT)
FIGS_SUBMIT_DIR = os.path.join(PROJECT_ROOT, 'figs-submit')
UTILS_DIR = os.path.join(FIGS_SUBMIT_DIR, 'utils')
if FIGS_SUBMIT_DIR not in sys.path:
    sys.path.insert(0, FIGS_SUBMIT_DIR)
if UTILS_DIR not in sys.path:
    sys.path.insert(0, UTILS_DIR)

print('PROJECT_ROOT =', PROJECT_ROOT)


import numpy as np
import os
import json
import sys

_THIS_DIR = os.path.dirname(os.path.abspath(__file__))
_LOCAL_UTILS = os.path.join(_THIS_DIR, "utils")
if _LOCAL_UTILS not in sys.path:
    sys.path.insert(0, _LOCAL_UTILS)

do_load = True
do_fig = True

freq_ratio = 1.0


_model_ = "all, 100-350, t3, ori"


_prepro_mode_ = "hilbert, no norm no baseline"



print(f"Fig2x3x: model = {_model_}, prepro_mode = {_prepro_mode_}, freq_ratio = {freq_ratio}")

os.makedirs(f"figs/Fig2x3x/{_model_}", exist_ok=True)


from scipy.stats import spearmanr, zscore


from rhythmicity import rhythmicity_bin
rhythmicity_bin.insert(0, 1.01)  # for digitize upper bound
num_bins = len(rhythmicity_bin) - 1


def collect_eta2_by_bin(all_bin_stat, legal_indices, pval_thresh, char="stim_order"):
    """
    returns: dict bin_idx -> np.array(eta2 values passing pval_thresh)
    """
    char_to_k = {"stim_order": 0, "interv_order": 1, "stim_direction": 2}
    k = char_to_k[char]

    eta2_by_bin = {}
    for bin_idx, bin_stat in all_bin_stat.items():
        if bin_stat is None:
            eta2_by_bin[bin_idx] = np.array([])
            continue
        eta2 = bin_stat[k][1]
        pval = bin_stat[k][2]
        idx = [i for i in legal_indices if pval[i] < pval_thresh]
        eta2_by_bin[bin_idx] = np.array([eta2[i] for i in idx], dtype=float)
    return eta2_by_bin


def theil_sen_slope(x, y):
    """Median of pairwise slopes; very robust. x,y are 1D arrays."""
    x = np.asarray(x); y = np.asarray(y)
    slopes = []
    n = len(x)
    for i in range(n):
        for j in range(i+1, n):
            if x[j] != x[i]:
                slopes.append((y[j] - y[i]) / (x[j] - x[i]))
    return np.median(slopes) if len(slopes) else np.nan


def quantile_trend_test(eta2_by_bin, tau=0.9, n_perm=5000, seed=0):
    """
    1) compute per-bin quantile q_tau
    2) compute robust slope of q_tau vs bin_y
    3) permutation test by shuffling bin labels of individual points
    """
    rng = np.random.default_rng(seed)

    # Build arrays of individual points (x) and their bin labels (b)
    xs = []
    bs = []
    for b, arr in eta2_by_bin.items():
        arr = arr[~np.isnan(arr)]
        xs.append(arr)
        bs.append(np.full(len(arr), b))
    xs = np.concatenate(xs) if len(xs) else np.array([])
    bs = np.concatenate(bs) if len(bs) else np.array([])

    # bins that actually have data
    uniq_bins = sorted([b for b in eta2_by_bin.keys() if len(eta2_by_bin[b]) > 0])
    if len(uniq_bins) < 3:
        return dict(q=None, slope=np.nan, rho=np.nan, p=np.nan)

    def compute_qs(bin_labels):
        qs = []
        for b in uniq_bins:
            vals = xs[bin_labels == b]
            qs.append(np.quantile(vals, tau))
        return np.array(qs)

    qs_obs = compute_qs(bs)
    slope_obs = theil_sen_slope(np.array(uniq_bins), qs_obs)
    rho_obs, _ = spearmanr(uniq_bins, qs_obs)

    # permutation: shuffle bin labels among points (keeps x distribution fixed)
    perm_slopes = []
    for _ in range(n_perm):
        bs_perm = rng.permutation(bs)
        qs_perm = compute_qs(bs_perm)
        perm_slopes.append(theil_sen_slope(np.array(uniq_bins), qs_perm))
    perm_slopes = np.array(perm_slopes)

    # two-sided p on slope
    p_perm = (np.sum(np.abs(perm_slopes) >= abs(slope_obs)) + 1) / (n_perm + 1)

    return dict(q=qs_obs, slope=slope_obs, rho=rho_obs, p=p_perm, bins=np.array(uniq_bins))


def bin_idx_to_y(bin_idx, num_bins):
    return num_bins - np.asarray(bin_idx)


def per_bin_quantile_points(eta2_by_bin_char, tau, num_bins):
    xs, ys, bs = [], [], []
    for bin_idx, arr in eta2_by_bin_char.items():
        arr = np.asarray(arr)
        arr = arr[~np.isnan(arr)]
        if len(arr) == 0:
            continue
        xs.append(np.quantile(arr, tau))
        ys.append(bin_idx_to_y(bin_idx, num_bins))
        bs.append(bin_idx)
    return np.array(xs), np.array(ys), np.array(bs)


def theil_sen_line(x, y):
    s = theil_sen_slope(x, y)
    b = np.median(y - s * x)
    return s, b


def draw_quantile_trend(ax, eta2_by_bin_char, tau, num_bins, color="black", label=None):
    qx, qy, bs = per_bin_quantile_points(eta2_by_bin_char, tau=tau, num_bins=num_bins)
    bin_idx = bs
    q = qx
    if len(bin_idx) < 2:
        return

    s, intercept = theil_sen_line(bin_idx, q)

    order = np.argsort(qy)
    ax.plot(qx[order], qy[order], "-o", color=color, lw=1.0, ms=3, zorder=5)

    bin_idx_line = np.linspace(bin_idx.min(), bin_idx.max(), 100)
    q_line = s * bin_idx_line + intercept

    y_line = bin_idx_to_y(bin_idx_line, num_bins)

    ax.plot(q_line, y_line, "--", color=color, lw=1.0, zorder=4, label=label)


def linear_trend_2d(mat):
    Y, X = [], []
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            if not np.isnan(mat[i, j]):
                Y.append(mat[i, j])
                X.append([i, j])
    Y = np.array(Y)
    X = sm.add_constant(np.array(X))  # intercept
    model = sm.OLS(Y, X).fit()
    return model


def diagonal_spearman(mat):
    """
    mat: 2D numpy array (rhythmicity x eta2)
    returns: rho_d
    """
    ys, x1s, x2s = [], [], []

    for i in range(mat.shape[0]):      # rhythmicity bin
        for j in range(mat.shape[1]):  # eta2 bin
            if not np.isnan(mat[i, j]):
                ys.append(mat[i, j])
                x1s.append(i)   # rhythmicity axis
                x2s.append(j)   # eta2 axis

    ys = np.array(ys)
    x1s = zscore(np.array(x1s))
    x2s = zscore(np.array(x2s))

    d = -x1s + x2s
    rho, _ = spearmanr(d, ys)
    return rho


def diagonal_spearman_perm(mat, n_perm=5000, seed=0):
    rng = np.random.default_rng(seed)

    ys, x1s, x2s = [], [], []
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            if not np.isnan(mat[i, j]):
                ys.append(mat[i, j])
                x1s.append(i)
                x2s.append(j)

    ys = np.array(ys)
    x1s = zscore(np.array(x1s))
    x2s = zscore(np.array(x2s))
    d = -x1s + x2s

    rho_obs, _ = spearmanr(d, ys)

    perm_rhos = []
    for _ in range(n_perm):
        y_perm = rng.permutation(ys)
        rho_perm, _ = spearmanr(d, y_perm)
        perm_rhos.append(rho_perm)

    perm_rhos = np.array(perm_rhos)
    p_perm = (np.sum(np.abs(perm_rhos) >= abs(rho_obs)) + 1) / (n_perm + 1)
    return rho_obs, p_perm


def get_rvals_and_yvals(num_bins, rhythmicity_bin):
    """
    rvals: actual rhythmicity values
    yvals: normalized positions in [0, 1]
    """
    rvals = np.asarray([rhythmicity_bin[i+1] for i in range(num_bins)], dtype=float)

    rmin = np.min(rvals)
    rmax = np.max(rvals)

    if np.isclose(rmax, rmin):
        yvals = np.zeros_like(rvals)
    else:
        yvals = (rvals - rmin) / (rmax - rmin)

    return rvals, yvals


def format_p(p):
    if p < 0.001:
        return "p < 0.001"
    elif p < 0.01:
        return "p < 0.01"
    elif p < 0.05:
        return "p < 0.05"
    else:
        return f"p = {p:.2f}"


def significance(pval):
    if pval < 0.0001:
        return "****"
    if pval < 0.001:
        return "***"
    elif pval < 0.01:
        return "**"
    elif pval < 0.05:
        return "*"
    else:
        return "n.s."


def print_ols_brief(model, x_names=("rhythmicity", "eta2")):
    params = model.params
    ses = model.bse
    pvals = model.pvalues

    print("OLS trend summary")
    print("-" * 40)
    print(f"N            = {int(model.nobs)}")
    print(f"Adj. R²      = {model.rsquared_adj:.3f}")
    print(f"F(2,{int(model.df_resid)}) = {model.fvalue:.2f}, p = {model.f_pvalue:.2e}")
    print()

    for i, name in enumerate(x_names, start=1):
        print(
            f"{name:<12s}: "
            f"β = {params[i]:+.4f}, "
            f"SE = {ses[i]:.4f}, "
            f"p = {pvals[i]:.2e} [{significance(pvals[i])}]"
        )


if __name__ == "__main__":

    char_name = {
        "stim_order": "stimulus order", 
        "interv_order": "interval order", 
        "stim_direction": "stimulus direction",
    }

    if do_load:

        activity_type = "neuronal"

        ### original data loader ###

        from activity_loader import load_activity_data_everything, load_metadata_and_parameters

        print("loading data...")

        if _model_[:3] == "all":
            if _model_ == "all, 100-350, t3, ori":
                data_parent_folder_ = 'models/stim_6/change_0_3-m_30_100_3/EIRNN_STSP-uap-dur_100-ioi_350-delay_1000-tvu/time_202512171812-turn_03'

            data_parent_folder = data_parent_folder_ + '/activity_all_pairs'
            npz_paths = [
                f'{data_parent_folder}/activity_rhy_ch0_all_pairs.npz',
                f'{data_parent_folder}/activity_rhy_ch3_all_pairs.npz',
                f'{data_parent_folder}/activity_arrhy_ch0_all_pairs.npz',
                f'{data_parent_folder}/activity_arrhy_ch3_all_pairs.npz',
            ]
            metadata_json_path = f'{data_parent_folder}/metadata_all_pairs.json'

        if _prepro_mode_ == "hilbert, no norm no baseline":
            do_normalization = True
            do_per_neuron_normalization = False
            do_align = False

        activity, efficacy, (syn_x, syn_u, base_u), input_activity, y_activity, targets, sample_seq, test_seq, sample_onset_raw, test_onset_raw = load_activity_data_everything(npz_paths)
        if activity_type == "neuronal":
            data = activity
        elif activity_type == "synaptic":
            data = efficacy

        for i in range(len(data)):
            print(f"condition {i}: data shape {activity[i].shape}")

        metadata, params, (stimulus_num, channel_num, time_step, fixation_duration, stimulus_duration, inter_onset_interval, sample_duration, delay_duration, response_duration, hidden_size, ei_list) = load_metadata_and_parameters(metadata_json_path)

        sample_onset = [so + fixation_duration // time_step for so in sample_onset_raw]  # fixation offset

        phase_orig = {
            "Fixation": (0, fixation_duration // time_step),
            "Sample": (fixation_duration // time_step, (fixation_duration + sample_duration) // time_step),     
            "Delay": ((fixation_duration + sample_duration) // time_step, (fixation_duration + sample_duration + delay_duration) // time_step),
            "Test": ((fixation_duration + sample_duration + delay_duration) // time_step, (fixation_duration + sample_duration + delay_duration + sample_duration) // time_step),
            "Response": ((fixation_duration + sample_duration + delay_duration + sample_duration) // time_step, (fixation_duration + sample_duration + delay_duration + sample_duration + response_duration) // time_step),
        } 

        fs = 1000 // time_step                          # sampling frequency in Hz
        stim_freq = 1000.0 / inter_onset_interval       # 350ms => 2.857 Hz

        stim_freq *= freq_ratio


        ### mask identity neurons ###

        from activity_loader import mask_identity_neurons

        illegal_indices_list = []
        for idx, single_conditional_data in enumerate(data):
            data[idx], illegal_indices = mask_identity_neurons(single_conditional_data)
            illegal_indices_list.append(illegal_indices)
            break
        illegal_indices = illegal_indices_list[0]
        legal_indices = sorted(list(set(range(hidden_size)) - set(illegal_indices)))


        ### normalization ###

        from activity_loader import normalize_data_zscore

        if do_normalization:
            data = normalize_data_zscore(data, do_per_neuron_normalization=do_per_neuron_normalization)
            print("data normalized using z-score method.")


        ### align

        from align import AlignConfig, align_data_full_trial

        if do_align:
            cfg = AlignConfig(align_mode="interval_pre", stim_len_steps=10, pre_len_proportion=0.5, warp_bins=30)
            aligned_data, phase_aligned = align_data_full_trial(data, sample_onset, phase_orig, cfg)
            data = aligned_data
            phase = phase_aligned
            print(f"data aligned into shape: {aligned_data.shape}, aligned phase indices: {phase_aligned}.")
        else:
            phase = phase_orig

        print(" ")


        ## CV_IOI per trial ##

        from rhythmicity import compute_fstim_cv

        print("computing CV_IOI and rhythmicity binning...")
        rhythmicity_l4 = []
        for so in sample_onset:
            fstim_cv = compute_fstim_cv(so, fs=fs)
            cv_ioi = fstim_cv['CV_IOI']
            rhythmicity = 1 - cv_ioi
            rhythmicity_l4.append(rhythmicity)

        trial_bin_l4 = []
        for rhythmicity in rhythmicity_l4:
            trial_bin = []
            for r in rhythmicity:
                bin_idx = np.digitize(r, rhythmicity_bin) - 1
                bin_idx = min(max(bin_idx, 0), num_bins - 1)
                trial_bin.append(bin_idx)
                # print(f"{r:.4f} -> bin {bin_idx} ({rhythmicity_bin[bin_idx]:.2f} ~ {rhythmicity_bin[bin_idx+1]:.2f})")
            trial_bin_l4.append(trial_bin)

        # bins, bin_trial_nums = np.unique(np.concatenate(trial_bin_l4), return_counts=True)
        # print(bin_trial_nums)

        binned_data = {}
        binned_sample_onset = {}
        binned_sample_seq = {}

        for bin_idx in range(num_bins):
            selected_trials = []
            selected_onsets = []
            selected_seqs = []
            for cond_idx in range(len(data)):
                for trial_idx in range(len(trial_bin_l4[cond_idx])):
                    if trial_bin_l4[cond_idx][trial_idx] == bin_idx:
                        selected_trials.append(data[cond_idx][trial_idx])
                        selected_onsets.append(sample_onset[cond_idx][trial_idx])
                        selected_seqs.append(sample_seq[cond_idx][trial_idx])
            binned_data[bin_idx] = np.array(selected_trials)
            binned_sample_onset[bin_idx] = np.array(selected_onsets)
            binned_sample_seq[bin_idx] = np.array(selected_seqs)

        print(" ")


        ## compute eta2 per bin ##

        from selectivity import SelectivityConfig, compute_eta2_for_trial_batch

        all_bin_stat = {}
        for bin_idx in range(num_bins):
            n_trials = len(binned_data[bin_idx])
            print(f"bin #{bin_idx}, trial num = {n_trials}")
            if n_trials == 0:
                all_bin_stat[bin_idx] = None
                continue

            cfg = SelectivityConfig(
                stim_len_steps=10,
                zscore=True,
                pre_len_steps=10,
                do_identity=True,
                residualize_id_by_pos=True,
                normal_alpha=0.05,
                use_shapiro=True,
                test_fdr_scope="global",
                sel_eta2_thr=0.05,
                sel_p_alpha=0.01,
                sel_use_fdr=True,
                out_dir=None,
                save_dpi=300,
                fontsize=11,
            )
            bin_stat = compute_eta2_for_trial_batch(binned_data[bin_idx], binned_sample_onset[bin_idx], binned_sample_seq[bin_idx], cfg)
            all_bin_stat[bin_idx] = bin_stat

        print(" ")


        ### save cached data ###

        import pickle

        with open(f"cache/Fig2x-m_{_model_}-{_prepro_mode_}-fr_{freq_ratio}.pkl", "wb") as f:
            pickle.dump((legal_indices, num_bins, all_bin_stat), f)

        print("cached data saved!")


    if do_load:

        ### IP EP PLV ###

        from rhythmicity import compute_rhythmicity_metrics

        print("computing IP, EP, PLV for each bin...")
        binned_ip_ep_plv = {}

        for bin_idx in range(num_bins):
            data, sample_onset, sample_seq = binned_data[bin_idx], binned_sample_onset[bin_idx], binned_sample_seq[bin_idx]
            if len(data) == 0:
                binned_ip_ep_plv[bin_idx] = None
                continue

            n_trials, n_neurons, n_time = data.shape
            sample_window = (phase_orig['Sample'][0], phase_orig['Sample'][1])

            if _prepro_mode_ == "hilbert, no norm no baseline" :
                baseline_window = None

            binned_ip_ep_plv[bin_idx] = {"IP": list(), "EP": list(), "PLV": list()}

            for neuron_idx in range(n_neurons):
                scores, curves = compute_rhythmicity_metrics(
                    data[:, neuron_idx, :], sample_onset,
                    fs=fs, 
                    target_freq=stim_freq, 
                    time_window=sample_window,
                    baseline_window=baseline_window,
                )
                binned_ip_ep_plv[bin_idx]["IP"].append(scores['induced_power_score'])
                binned_ip_ep_plv[bin_idx]["EP"].append(scores['evoked_power_score'])
                binned_ip_ep_plv[bin_idx]["PLV"].append(scores['plv_score'])
            print(f"bin {bin_idx} done.")

        print(" ")


        ### save cached data ###

        import pickle

        with open(f"cache/Fig3x-m_{_model_}-{_prepro_mode_}-fr_{freq_ratio}.pkl", "wb") as f:
            pickle.dump((legal_indices, pval_thresh, num_bins, all_bin_stat, binned_ip_ep_plv), f)

        print("cached data saved!")


    if do_fig:

        #####################################################################

        ### load cached data ###

        import pickle

        with open(f"cache/Fig3x-m_{_model_}-{_prepro_mode_}-fr_{freq_ratio}.pkl", "rb") as f:
            legal_indices, pval_thresh, num_bins, all_bin_stat, binned_ip_ep_plv = pickle.load(f)

        print("cached data loaded!")

        pval_thresh = 0.05

        
        ### bin IP EP PLV ###

        print("binning IP, EP, PLV by selectivity eta2...")

        num_eta2_bins = num_bins
        eta2_bin_step = 1.0 / num_eta2_bins

        binned__stim_order_binned__ip_ep_plv, binned__interv_order_binned__ip_ep_plv, binned__stim_direction_binned__ip_ep_plv = {}, {}, {}
        binned__stim_order_binned__eta2, binned__interv_order_binned__eta2, binned__stim_direction_binned__eta2 = {}, {}, {}
        for bin_idx in range(num_bins):
            bin_stat = all_bin_stat[bin_idx]
            if bin_stat is None:
                for eta2_bin_idx in range(num_eta2_bins):
                    key = (bin_idx, eta2_bin_idx)
                    binned__stim_order_binned__ip_ep_plv[key] = {"IP": [np.nan], "EP": [np.nan], "PLV": [np.nan]}
                    binned__interv_order_binned__ip_ep_plv[key] = {"IP": [np.nan], "EP": [np.nan], "PLV": [np.nan]}
                    binned__stim_direction_binned__ip_ep_plv[key] = {"IP": [np.nan], "EP": [np.nan], "PLV": [np.nan]}
                continue
            stim_order_eta2, stim_order_pval = bin_stat[0][1], bin_stat[0][2]
            interv_order_eta2, interv_order_pval = bin_stat[1][1], bin_stat[1][2]
            stim_direction_eta2, stim_direction_pval = bin_stat[2][1], bin_stat[2][2]

            stim_order_indices = [idx for idx in legal_indices if stim_order_pval[idx] < pval_thresh]
            interv_order_indices = [idx for idx in legal_indices if interv_order_pval[idx] < pval_thresh]
            stim_direction_indices = [idx for idx in legal_indices if stim_direction_pval[idx] < pval_thresh]

            for eta2_bin_idx in range(num_eta2_bins):
                eta2_bin_lower = eta2_bin_idx * eta2_bin_step
                eta2_bin_upper = (eta2_bin_idx + 1) * eta2_bin_step

                # stim order
                key = (bin_idx, eta2_bin_idx)
                selected_indices = [idx for idx in stim_order_indices if stim_order_eta2[idx] >= eta2_bin_lower and stim_order_eta2[idx] < eta2_bin_upper]
                if len(selected_indices) == 0:
                    binned__stim_order_binned__ip_ep_plv[key] = {"IP": [np.nan], "EP": [np.nan], "PLV": [np.nan]}
                else:
                    binned__stim_order_binned__ip_ep_plv[key] = {"IP": [], "EP": [], "PLV": []}
                    binned__stim_order_binned__ip_ep_plv[key]["IP"].extend([binned_ip_ep_plv[bin_idx]["IP"][idx] for idx in selected_indices])
                    binned__stim_order_binned__ip_ep_plv[key]["EP"].extend([binned_ip_ep_plv[bin_idx]["EP"][idx] for idx in selected_indices])
                    binned__stim_order_binned__ip_ep_plv[key]["PLV"].extend([binned_ip_ep_plv[bin_idx]["PLV"][idx] for idx in selected_indices])

                # interv order
                key = (bin_idx, eta2_bin_idx)
                selected_indices = [idx for idx in interv_order_indices if interv_order_eta2[idx] >= eta2_bin_lower and interv_order_eta2[idx] < eta2_bin_upper]
                if len(selected_indices) == 0:
                    binned__interv_order_binned__ip_ep_plv[key] = {"IP": [np.nan], "EP": [np.nan], "PLV": [np.nan]}
                else:
                    binned__interv_order_binned__ip_ep_plv[key] = {"IP": [], "EP": [], "PLV": []}
                    binned__interv_order_binned__ip_ep_plv[key]["IP"].extend([binned_ip_ep_plv[bin_idx]["IP"][idx] for idx in selected_indices])
                    binned__interv_order_binned__ip_ep_plv[key]["EP"].extend([binned_ip_ep_plv[bin_idx]["EP"][idx] for idx in selected_indices])
                    binned__interv_order_binned__ip_ep_plv[key]["PLV"].extend([binned_ip_ep_plv[bin_idx]["PLV"][idx] for idx in selected_indices])
                # stim direction
                key = (bin_idx, eta2_bin_idx)
                selected_indices = [idx for idx in stim_direction_indices if stim_direction_eta2[idx] >= eta2_bin_lower and stim_direction_eta2[idx] < eta2_bin_upper]
                if len(selected_indices) == 0:
                    binned__stim_direction_binned__ip_ep_plv[key] = {"IP": [np.nan], "EP": [np.nan], "PLV": [np.nan]}
                else:
                    binned__stim_direction_binned__ip_ep_plv[key] = {"IP": [], "EP": [], "PLV": []}
                    binned__stim_direction_binned__ip_ep_plv[key]["IP"].extend([binned_ip_ep_plv[bin_idx]["IP"][idx] for idx in selected_indices])
                    binned__stim_direction_binned__ip_ep_plv[key]["EP"].extend([binned_ip_ep_plv[bin_idx]["EP"][idx] for idx in selected_indices])
                    binned__stim_direction_binned__ip_ep_plv[key]["PLV"].extend([binned_ip_ep_plv[bin_idx]["PLV"][idx] for idx in selected_indices])

        print(" ")


        ### plot binned IP EP PLV ###

        char_display_order = ["stim_order", "interv_order", "stim_direction"]
        metric_display_order = ["EP", "PLV"]

        print("plotting fig3x...")

        rvals, rhythmicity_norm = get_rvals_and_yvals(num_bins, rhythmicity_bin)

        import matplotlib
        import matplotlib.pyplot as plt
        import statsmodels.api as sm

        font_path = "utils/Arial"
        font_files = matplotlib.font_manager.findSystemFonts(fontpaths=font_path)
        for file in font_files:
            matplotlib.font_manager.fontManager.addfont(file)

        plt.rcParams["font.family"] = "Arial"
        plt.rcParams["font.size"] = 8

        fig = plt.figure(figsize=(9, 5))
        gs = fig.add_gridspec(
            len(metric_display_order), len(char_display_order), 
            left=0.08, right=0.96, bottom=0.08, top=0.92, 
            height_ratios=[1.0] * len(metric_display_order), hspace=0.5,
            width_ratios=[1.0] * len(char_display_order), wspace=0.3,
        )

        binned__char_binned__metrics = {
            "stim_order": binned__stim_order_binned__ip_ep_plv, 
            "interv_order": binned__interv_order_binned__ip_ep_plv, 
            "stim_direction": binned__stim_direction_binned__ip_ep_plv,
        }
        for char_idx, char in enumerate(char_display_order):
            for metric_idx, metric in enumerate(metric_display_order):
                print(" ")
                print(f"plotting char {char}, metric {metric}...")
                print(" ")
                mat = np.zeros((num_bins, num_eta2_bins))
                for bin_idx in range(num_bins):
                    for eta2_bin_idx in range(num_eta2_bins):
                        key = (bin_idx, eta2_bin_idx)
                        metric_values = binned__char_binned__metrics[char][key][metric]
                        metric_mean = np.nanmean(metric_values)
                        mat[bin_idx, eta2_bin_idx] = metric_mean
                
                model = linear_trend_2d(mat)

                rho_d, p_d = diagonal_spearman_perm(mat, n_perm=5000)
                adj_r2 = model.rsquared_adj


                ax = fig.add_subplot(gs[metric_idx, char_idx])
                im = ax.imshow(
                    np.fliplr(mat.T),
                    origin='lower',
                    aspect=1,
                    extent=[0, 1, 0, 1],
                    cmap='jet',
                )
                annotation_text = (
                    f"diag ρ = {rho_d:.2f}"
                    f"\n{format_p(p_d)}"
                    # f\nAdj. R² = {adj_r2:.2f}"
                )

                ax.text(
                    0.98, 0.02,
                    annotation_text,
                    transform=ax.transAxes,
                    ha="right", va="bottom",
                    fontsize=7,
                    bbox=dict(
                        facecolor="white",
                        edgecolor="none",
                        alpha=0.6,
                        boxstyle="round,pad=0.2"
                    )
                )
                ax.set_title(f"{metric}")
                ax.set_xlabel("normalized rhythmicity")
                ax.set_xticks(np.linspace(0, 1, 5))
                ax.set_xticklabels(["0.0", "0.25", "0.5", "0.75", "1.0"])
                ax.set_ylabel(f"η² ({char_name[char]})")
                ax.set_yticks(np.linspace(0, 1, 5))
                ax.set_yticklabels(["0.0", "0.25", "0.5", "0.75", "1.0"])
                fig.colorbar(im, ax=ax, shrink=1)
        
        fig.savefig(f"figs/Fig2x3x/{_model_}/Fig3x-fr_{freq_ratio}.pdf", dpi=300)
        print("fig3x saved!")
    